Chapter 5 - Pretraining on Unlabeled Data

5.1.1 - Using GPT to generate text

In [7]:
import torch

In [8]:
from previous_chapters import GPTModel

In [9]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

In [10]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

In [11]:
import tiktoken
from previous_chapters import generate_text_simple

In [12]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

In [13]:
def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

In [14]:
start_context = "Every effort moves you"

In [15]:
tokenizer = tiktoken.get_encoding("gpt2")


In [16]:
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

In [17]:
token_ids_to_text(token_ids, tokenizer)

'Every effort moves you rentingetic wasnم refres RexMeCHicular stren'

5.1.2 - Calculating the text generation loss: cross-entropy and perplexity

In [18]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",
                        [1107,  588, 11311]]) #  " really like chocolate"]

In [19]:
with torch.no_grad():
    logits = model(inputs)

In [20]:
probas = torch.softmax(logits, dim=-1)

In [21]:
probas.shape

torch.Size([2, 3, 50257])

In [22]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)

In [23]:
token_ids

tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])

In [24]:
token_ids_to_text(targets[0], tokenizer)

' effort moves you'

In [25]:
token_ids_to_text(token_ids[0].flatten(), tokenizer)

' Armed heNetflix'

In [26]:
token_ids[0].flatten().shape

torch.Size([3])

In [27]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
target_probas_1

tensor([7.4540e-05, 3.1061e-05, 1.1563e-05])

In [28]:
text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
target_probas_2

tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])

In [29]:
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
log_probas

tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561])

In [30]:
avg_log_probas = torch.mean(log_probas)
avg_log_probas

tensor(-10.7940)

In [31]:
neg_avg_log_probas = avg_log_probas * -1
neg_avg_log_probas

tensor(10.7940)

In [32]:
logits.shape

torch.Size([2, 3, 50257])

In [33]:
targets.shape

torch.Size([2, 3])

In [34]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

In [35]:
logits_flat.shape, targets_flat.shape

(torch.Size([6, 50257]), torch.Size([6]))

In [36]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
loss

tensor(10.7940)

In [37]:
perplexity = torch.exp(loss)
perplexity

tensor(48725.8203)

5.1.3 - Calculating the training and validation set losses

In [38]:
import os
import requests

In [39]:
file_path = "the-verdict.txt"
url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"

In [40]:
if not os.path.exists(file_path):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    text_data = response.text
    with open(file_path, "w", encoding="utf-8") as file:
        file.write(text_data)
else:
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()

In [41]:
print(text_data[:99])

I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [42]:
print(text_data[-99:])

it for me! The Strouds stand alone, and happen once--but there's no exterminating our kind of art."


In [43]:
total_characters = len(text_data)
total_tokens = len(tokenizer.encode(text_data))
total_characters, total_tokens

(20479, 5145)

In [44]:
from previous_chapters import create_dataloader_v1

In [45]:
train_ratio = 0.90
split_idx = int(train_ratio * len(text_data))
train_data = text_data[:split_idx]
val_data = text_data[split_idx:]

In [46]:
torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [47]:
# Sanity check

if total_tokens * (train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the training loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "increase the `training_ratio`")

if total_tokens * (1-train_ratio) < GPT_CONFIG_124M["context_length"]:
    print("Not enough tokens for the validation loader. "
          "Try to lower the `GPT_CONFIG_124M['context_length']` or "
          "decrease the `training_ratio`")

In [48]:
for x,y in train_loader:
    print(x.shape, y.shape)

torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])


In [49]:
for x, y in val_loader:
    print(x.shape, y.shape)

torch.Size([2, 256]) torch.Size([2, 256])


In [50]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

train_tokens, val_tokens

(4608, 512)

In [51]:
train_tokens + val_tokens

5120

In [70]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    print(f"CALC_LOSS_BATCH1: logits before {logits.shape}, target_batch before {target_batch.shape}")
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
    print(f"CALC_LOSS_BATCH2: logits after {logits.flatten(0, 1).shape}, target_batch after {target_batch.flatten().shape}")
    return loss

In [71]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches


In [68]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    major, minor = map(int, torch.__version__.split('.')[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print(f"Using {device} device.")

Using cuda device.


In [55]:
model.to(device)

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features

In [72]:
torch.manual_seed(123)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

CALC_LOSS_BATCH1: logits before torch.Size([2, 256, 50257]), target_batch before torch.Size([2, 256])
CALC_LOSS_BATCH2: logits after torch.Size([512, 50257]), target_batch after torch.Size([512])
CALC_LOSS_BATCH1: logits before torch.Size([2, 256, 50257]), target_batch before torch.Size([2, 256])
CALC_LOSS_BATCH2: logits after torch.Size([512, 50257]), target_batch after torch.Size([512])
CALC_LOSS_BATCH1: logits before torch.Size([2, 256, 50257]), target_batch before torch.Size([2, 256])
CALC_LOSS_BATCH2: logits after torch.Size([512, 50257]), target_batch after torch.Size([512])
CALC_LOSS_BATCH1: logits before torch.Size([2, 256, 50257]), target_batch before torch.Size([2, 256])
CALC_LOSS_BATCH2: logits after torch.Size([512, 50257]), target_batch after torch.Size([512])
CALC_LOSS_BATCH1: logits before torch.Size([2, 256, 50257]), target_batch before torch.Size([2, 256])
CALC_LOSS_BATCH2: logits after torch.Size([512, 50257]), target_batch after torch.Size([512])
CALC_LOSS_BATCH1: lo

In [74]:
train_loss, val_loss

(10.987583266364204, 10.98110580444336)